In [108]:
import os
import fiona
import rasterio
import rasterio.mask
import rioxarray as rxr
import glob
import geopandas as gpd
import pandas as pd
from rasterio.warp import calculate_default_transform, reproject, Resampling
import numpy as np

In [101]:
dir1 = "C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1"
dir2 = "C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2"

task1 = []
task2 = []
for root, dirs, files in os.walk(dir1):
    for directory in dirs:
        task1.append(os.path.join(root, directory))
        
for root, dirs, files in os.walk(dir2):
    for directory in dirs:
        task2.append(os.path.join(root, directory))
print(task1)
print("-------------------------------")
print(task2)

['C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\\20230405', 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\\20240315', 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\\20240418', 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\\20250404', 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\\20250501']
-------------------------------
['C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2\\20230405', 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2\\20240315', 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2\\20240418', 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2\\20250404', 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2\\20250501']


In [102]:
def swe_units(task_dir):
    for root, dirs, files in os.walk(task_dir):
        for directory in dirs:
            date = os.path.basename(directory)

            if os.path.isfile(glob.glob(os.path.join(root,directory, "*EB_swe*"))[0]):
                with rasterio.open(glob.glob(os.path.join(root,directory, "*EB_swe*"))[0]) as src:
                    data = src.read(1, masked=True)
                    profile = src.profile
                    profile.update(dtype=rasterio.float32, nodata=-9999)
                    outfile = os.path.join(root,directory, f"{date}_EB_cm.tif")
                    EB_cm = data * 2.54
                with rasterio.open(outfile, "w", **profile) as dst:
                    dst.write(EB_cm.astype("float32"), 1)
            else:
                print("No energy balance raster found")
                
            if os.path.isfile(glob.glob(os.path.join(root,directory, "*TI_swe*"))[0]):
                with rasterio.open(glob.glob(os.path.join(root,directory, "*TI_swe*"))[0]) as src:
                    data = src.read(1, masked=True)
                    profile = src.profile
                    profile.update(dtype=rasterio.float32, nodata=-9999)
                    outfile = os.path.join(root,directory, f"{date}_TI_cm.tif")
                    TI_cm = data * 2.54
                with rasterio.open(outfile, "w", **profile) as dst:
                    dst.write(TI_cm.astype("float32"), 1)
            else:
                print("No temperature index raster found") 
            
            if os.path.isfile(glob.glob(os.path.join(root,directory, "*swed*"))[0]):
                with rasterio.open(glob.glob(os.path.join(root,directory, "*swed*"))[0]) as src:
                    data = src.read(1, masked=True)
                    profile = src.profile
                    profile.update(dtype=rasterio.float32, nodata=-9999)
                    outfile = os.path.join(root,directory, f"{date}_SnowModel_cm.tif")
                    SM_cm = data * 100
                with rasterio.open(outfile, "w", **profile) as dst:
                    dst.write(SM_cm.astype("float32"), 1)
            else:
                print("No SnowModel raster found") 
            
            if os.path.isfile(glob.glob(os.path.join(root,directory, "*specific_mass*"))[0]):
                with rasterio.open(glob.glob(os.path.join(root,directory, "*specific_mass*"))[0]) as src:
                    data = src.read(1, masked=True)
                    profile = src.profile
                    profile.update(dtype=rasterio.float32, nodata=-9999)
                    outfile = os.path.join(root,directory, f"{date}_iSnobal_cm.tif")
                    iSnobal_cm = data / 10
                with rasterio.open(outfile, "w", **profile) as dst:
                    dst.write(iSnobal_cm.astype("float32"), 1)
            else:
                print("No iSnobal raster found") 
                
                
                        

In [103]:
swe_units("C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2")
swe_units("C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1")

In [107]:
basin_shp = "C:/Users/RDCRLSMC/Desktop/GIS/basin/basin_outline.shp"
with fiona.open(basin_shp, "r") as shapefile:
    shapes = [feature["geometry"] for feature in shapefile]

In [119]:

for dir in task1:
    print(dir)

C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20230405
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20240315
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20240418
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20250404
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20250501


In [125]:
filename = "C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2/20250501/EB_swe_2025_05_01T0000_2025_05_01T2400_100m.tif"
name, ext = os.path.splitext(filename)
new_name = os.path.join(f"{name}_MC" + ext)
print(new_name)

C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2/20250501/EB_swe_2025_05_01T0000_2025_05_01T2400_100m_MC.tif


In [126]:
def clip_basin(parent_dir):
    target_string = 'cm'
    for dir in parent_dir:
        for root, dirs, files in os.walk(dir):
            #print(files)
            for filename in files:
                if target_string in filename:
                    name, ext = os.path.splitext(filename)
                    new_name = os.path.join(f"{name}_MC" + ext)
                    out_path = os.path.join(dir, new_name)
                    if os.path.exists(out_path):
                        print(f"Skipping existing raster: {name}")
                    else:       
                        with rasterio.open(os.path.join(dir, filename)) as src:
                            out_image, out_transform = rasterio.mask.mask(src, shapes, crop=True)
# Copy the old profile and update it with new metadata
                            profile = src.profile
                            profile.update({
                            "driver": "GTiff",
                            "height": out_image.shape[1],
                            "width": out_image.shape[2],
                            "transform": out_transform, # <-- Use the new transform from the mask operation
                            "nodata": -9999 # Explicitly set nodata if not already
                             })
                        with rasterio.open(out_path, "w", **profile) as dest:
                                dest.write(out_image)
                    

In [127]:
clip_basin(task1)
clip_basin(task2)

In [128]:
def task_process(task_dir):
    swedata = {}
    for root, dirs, files in os.walk(task_dir):
        for directory in dirs:
            date = os.path.basename(directory)
            swedata[date] = {}
            swedata[date]["Energy Balance"] = glob.glob(os.path.join(root,directory, "*EB_cm_MC*"))[0]
            swedata[date]["Temperature Index"] = glob.glob(os.path.join(root,directory, "*TI_cm_MC*"))[0]
            swedata[date]["SnowModel"] = glob.glob(os.path.join(root,directory, "*SnowModel_cm_MC*"))[0]
            swedata[date]["iSnobal"] = glob.glob(os.path.join(root,directory, "*iSnobal_cm_MC*"))[0]
    return swedata

In [129]:
task2 = task_process("C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2")
task1 = task_process("C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1")
print(task2)
print("----------------------------------------------")
print(task1)

{'20230405': {'Energy Balance': 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2\\20230405\\20230405_EB_cm_MC.tif', 'Temperature Index': 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2\\20230405\\20230405_TI_cm_MC.tif', 'SnowModel': 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2\\20230405\\20230405_SnowModel_cm_MC.tif', 'iSnobal': 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2\\20230405\\20230405_iSnobal_cm_MC.tif'}, '20240315': {'Energy Balance': 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2\\20240315\\20240315_EB_cm_MC.tif', 'Temperature Index': 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2\\20240315\\20240315_TI_cm_MC.tif', 'SnowModel': 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2\\20240315\\20240315_SnowModel_cm_MC.tif', 'iSnobal': 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2\\20240315\\20240315_iSnobal_cm_MC.tif'}, '20240418': {'Energy Balance': 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2\\20240

In [132]:
task1

{'20230405': {'Energy Balance': 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\\20230405\\20230405_EB_cm_MC.tif',
  'Temperature Index': 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\\20230405\\20230405_TI_cm_MC.tif',
  'SnowModel': 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\\20230405\\20230405_SnowModel_cm_MC.tif',
  'iSnobal': 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\\20230405\\20230405_iSnobal_cm_MC.tif'},
 '20240315': {'Energy Balance': 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\\20240315\\20240315_EB_cm_MC.tif',
  'Temperature Index': 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\\20240315\\20240315_TI_cm_MC.tif',
  'SnowModel': 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\\20240315\\20240315_SnowModel_cm_MC.tif',
  'iSnobal': 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\\20240315\\20240315_iSnobal_cm_MC.tif'},
 '20240418': {'Energy Balance': 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SW

In [131]:
for date, models in task1.items():
    
    # Loop through each model for the current date
        for model_name, raster in models.items():
            print(raster)

C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20230405\20230405_EB_cm_MC.tif
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20230405\20230405_TI_cm_MC.tif
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20230405\20230405_SnowModel_cm_MC.tif
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20230405\20230405_iSnobal_cm_MC.tif
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20240315\20240315_EB_cm_MC.tif
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20240315\20240315_TI_cm_MC.tif
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20240315\20240315_SnowModel_cm_MC.tif
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20240315\20240315_iSnobal_cm_MC.tif
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20240418\20240418_EB_cm_MC.tif
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20240418\20240418_TI_cm_MC.tif
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20240418\20240418_SnowModel_cm_MC.tif
C:/Users/RDCRLSMC/Deskt

In [97]:
def calculate_total_weighted_sum(dictionary,  csv_file_path, tasknumber):
    """
    Calculates the sum of all raster pixel values, weighted by the pixel area.

    Args:
        filename (str): The path to the input raster file.

    Returns:
        float: The total weighted sum (e.g., volume if pixel values are height in meters).
    """
    
    results_list = []
    
    for date, models in dictionary.items():
    
    # Loop through each model for the current date
        for model_name, raster in models.items():
    
            with rasterio.open(raster) as src:
                # Read the first band as a masked numpy array to handle NoData values
                array = src.read(1, masked=True)
                non_na_count_masked = array.count()
                # Get metadata for pixel size (resolution)
                # Assumes the CRS units are appropriate for area calculation (e.g., meters)
                pixel_width = src.res[0]
                pixel_height = src.res[1]
                pixel_area = abs(pixel_width * pixel_height) # Use abs() as y resolution is often negative
        
       
                # Multiply each pixel value by the pixel area (element-wise multiplication with numpy)
                swe_m = array / 100
                weighted_volumes = swe_m * pixel_area
        
                # Sum all the weighted values (only valid pixels are summed due to masking)
                total_sum = weighted_volumes.sum()
                
                row = {
                    "task_number": tasknumber,
                    "date": date,
                    "pixel width (m)": pixel_width,
                    "pixel height (m)": pixel_height,
                    "pixel area (m2)": pixel_area,
                    "pixel count": non_na_count_masked,
                    "total area": pixel_area * non_na_count_masked,
                    "model name": model_name,
                    "total swe (m3)": total_sum
                }
                
                results_list.append(row)
                    
    df = pd.DataFrame(results_list)
    df.to_csv(csv_file_path, index=False)
        


In [100]:
calculate_total_weighted_sum(task2, "C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task2/Task2_totalSWE.csv", 2)
calculate_total_weighted_sum(task1, "C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1/Task1_totalSWE.csv", 1)

## The watershed boundaries are not exactly the same, so I want to do the same thing as above but only sum SWE where a pixel is TRUE for all rasters

In [159]:
print(task1)


{'20230405': {'Energy Balance': 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\\20230405\\20230405_EB_cm_MC.tif', 'Temperature Index': 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\\20230405\\20230405_TI_cm_MC.tif', 'SnowModel': 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\\20230405\\20230405_SnowModel_cm_MC.tif', 'iSnobal': 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\\20230405\\20230405_iSnobal_cm_MC.tif'}, '20240315': {'Energy Balance': 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\\20240315\\20240315_EB_cm_MC.tif', 'Temperature Index': 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\\20240315\\20240315_TI_cm_MC.tif', 'SnowModel': 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\\20240315\\20240315_SnowModel_cm_MC.tif', 'iSnobal': 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\\20240315\\20240315_iSnobal_cm_MC.tif'}, '20240418': {'Energy Balance': 'C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\\20240

In [157]:
for key in task1.keys():
    for model, raster in task1[key].items():
         print(key, model)
         print(raster)


20230405 Energy Balance
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20230405\20230405_EB_cm_MC.tif
20230405 Temperature Index
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20230405\20230405_TI_cm_MC.tif
20230405 SnowModel
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20230405\20230405_SnowModel_cm_MC.tif
20230405 iSnobal
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20230405\20230405_iSnobal_cm_MC.tif
20240315 Energy Balance
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20240315\20240315_EB_cm_MC.tif
20240315 Temperature Index
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20240315\20240315_TI_cm_MC.tif
20240315 SnowModel
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20240315\20240315_SnowModel_cm_MC.tif
20240315 iSnobal
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20240315\20240315_iSnobal_cm_MC.tif
20240418 Energy Balance
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20240418\20240418_EB_cm_MC.tif
2024

In [177]:
TI = "C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1/20230405/20230405_TI_cm_MC.tif"
EB = "C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1/20230405/20230405_EB_cm_MC.tif"

with rasterio.open(TI) as src1, rasterio.open(EB) as src2:
    print(src1.crs, src2.crs)
    transform1, transform2 = src1.transform, src2.transform
    shape1, shape2 = src1.shape, src2.shape
    
    print(f"\nShape 1: {shape1} (Height, Width)")
    print(f"Shape 2: {shape2} (Height, Width)")
    
    print(f"\nTransform 1: {transform1}")
    print(f"Transform 2: {transform2}")


EPSG:32611 EPSG:32611

Shape 1: (465, 364) (Height, Width)
Shape 2: (465, 364) (Height, Width)

Transform 1: | 100.00, 0.00, 572400.00|
| 0.00,-100.00, 4878000.00|
| 0.00, 0.00, 1.00|
Transform 2: | 100.00, 0.00, 572400.00|
| 0.00,-100.00, 4878000.00|
| 0.00, 0.00, 1.00|


In [193]:

def rasters_align(task_dictionary):
    # 1. & 2. Align rasters (Simplified: Assuming one is reference)
    # Re-sample all to match the first file (reference)
    
    for date, models in task_dictionary.items():
        model_names = list(models.keys())
        reference_model = model_names[0]
        reference_path = models[reference_model]
        resample_rasters = model_names[1:]
        
        print(f"Reference path is {reference_path}")
        for raster in resample_rasters:
           print(models[raster])
        
        # Open the reference raster to get its properties
        with rasterio.open(reference_path) as ref:
            ref_data = ref.read(1, masked=True)
            profile = ref.profile
            ref_transform = ref.transform
            ref_crs = ref.crs
            
        
        for model_name in resample_rasters:
            source_path = models[model_name]
            base, ext = os.path.splitext(source_path)
            output_path = f"{base}_resampled{ext}"
            
            with rasterio.open(source_path) as src:
                
                if (src.crs == ref_crs and
                    src.transform == ref_transform and
                    src.shape == ref.shape):
                    print(f" -> Skipping resampling, '{model_name}' is already aligned.")
                else:
                    model_data = src.read(1, masked=True)
                    model_transform = src.transform
                    model_crs = src.crs
                    
                    if model_crs is None:
                        model_crs = ref_crs
                
    
                    reprojected_model = np.empty(ref_data.shape, dtype=np.float32)
                    reproject(
                            source=model_data,
                            destination=reprojected_model,
                            src_transform=model_transform,
                            src_crs=model_crs,
                            dst_transform=ref_transform,
                            dst_crs=ref_crs,
                            resampling=Resampling.nearest
                        )
            
                    out_profile = profile.copy()
            
                    with rasterio.open(output_path, "w", **out_profile) as dst:
                        dst.write(reprojected_model.astype(np.float32), 1)

In [194]:
rasters_align(task1)

Reference path is C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20230405\20230405_EB_cm_MC.tif
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20230405\20230405_TI_cm_MC.tif
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20230405\20230405_SnowModel_cm_MC.tif
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20230405\20230405_iSnobal_cm_MC.tif
 -> Skipping resampling, 'Temperature Index' is already aligned.
 -> Skipping resampling, 'iSnobal' is already aligned.
Reference path is C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20240315\20240315_EB_cm_MC.tif
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20240315\20240315_TI_cm_MC.tif
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20240315\20240315_SnowModel_cm_MC.tif
C:/Users/RDCRLSMC/Desktop/SIRO/Model_Outputs/SWE/Task1\20240315\20240315_iSnobal_cm_MC.tif
 -> Skipping resampling, 'Temperature Index' is already aligned.
 -> Skipping resampling, 'iSnobal' is already aligned.
Reference pa